# Site 9 discrepancy: why DASM and rate ratios disagree

DASM predicts strong purifying selection at IGHV1 site 9 A→G (log SF ≈ −1.03),
but the Thrifty-based observed/expected rate ratio is near zero (neutral).

This notebook shows the discrepancy arises because **IGHV1-18** has atypically
positive selection at this site and contributes disproportionately to both
observed and expected counts due to higher neutral mutability, pulling the
aggregate ratio toward its individual (positive-selection) value.

The median-based aggregation across V genes used for DASM is robust to such outliers.


In [1]:
import os
import numpy as np
import pandas as pd
from types import SimpleNamespace

from utils import (
    sort_antibody_sites, add_germline_information,
    load_and_process_dnsm_data, load_and_process_dasm_data,
    GERMLINE_PATH_DICTIONARY,
)
from dnsmex.neutral_mutability import CachedNeutralMutabilityDataset
from rates_analysis_util import add_mutation_counts_per_branch_for_branch_length

DATASETS = ['v1jaffe', 'v1tang']
NUMBERING_SCHEME = 'chothia'
BRANCH_LENGTH_METHOD = 'mutation_frequency'
BRANCH_LENGTH_SCALE_FACTOR = 1.60
PSEUDOCOUNT = 0.5

TARGET_SITE = '9'
TARGET_V_FAMILY = 'IGHV1'
TARGET_PARENT_AA = 'A'
TARGET_CHILD_AA = 'G'


## 1. DASM selection factors per V gene at site 9

Load DASM selection factors from the Rodriguez dataset and break down
by V gene to show that most IGHV1 genes predict strong purifying selection.


In [2]:
dasm_compare_model_name = "dasm_4m-v1jaffeCC+v1tangCC-joint"
dasm_compare_dataset_name = "v1rodriguez"

site_sub_probs_df, pcp_df_dasm, aa_df = load_and_process_dasm_data(
    model_name=dasm_compare_model_name,
    dataset_name=dasm_compare_dataset_name,
    numbering_scheme=NUMBERING_SCHEME,
)


Adding one_mutation_away column (vectorized)...


In [3]:
# Filter to site 9, IGHV1, A parent, G target, standard filters
dasm_site9 = aa_df[
    (aa_df['site'] == TARGET_SITE) &
    (aa_df['v_family'] == TARGET_V_FAMILY) &
    (aa_df['parent_aa'] == TARGET_PARENT_AA) &
    (aa_df['selection_factor_target_aa'] == TARGET_CHILD_AA) &
    (aa_df['is_germline_codon'] == True) &
    (aa_df['one_mutation_away'] == True) &
    (aa_df['depth'] == 2)
].copy()

by_vgene_dasm = dasm_site9.groupby('v_gene').agg(
    n_pcps=('pcp_index', 'count'),
    median_log_sf=('log_selection_factor', 'median'),
    mean_log_sf=('log_selection_factor', 'mean'),
).sort_values('n_pcps', ascending=False)

print(f"Overall median log SF: {dasm_site9['log_selection_factor'].median():.4f}")
print(f"\nPer-V-gene DASM selection factors at site 9 (IGHV1 A→G):")
print(by_vgene_dasm.to_string())


Overall median log SF: -1.0276

Per-V-gene DASM selection factors at site 9 (IGHV1 A→G):
               n_pcps  median_log_sf  mean_log_sf
v_gene                                           
IGHV1-69*01       539      -0.972909    -0.853630
IGHV1-2*02        347      -1.159822    -1.052580
IGHV1-18*01       266       0.942035     0.872444
IGHV1-46*01       217      -1.134124    -1.072266
IGHV1-3*01        213      -1.037322    -0.965880
IGHV1-2*06        120      -1.201028    -1.110834
IGHV1-24*01       120      -1.189354    -1.145563
IGHV1-8*01        119      -1.334214    -1.220090
IGHV1-69*15        65      -1.157260    -1.127984
IGHV1-18*04        61       1.026230     0.992864
IGHV1-69-2*01      40      -1.066032    -0.946650
IGHV1-69*10        34      -0.926265    -0.718556
IGHV1-69*02        33      -1.052612    -0.912952
IGHV1-69*04        31      -1.059219    -0.979729
IGHV1-2*04         29      -0.973707    -0.998518
IGHV1-46*03        28      -1.052731    -0.917147
IGHV1-3*04 

## 2. Observed mutation counts per V gene at site 9

Load observed data using `load_and_process_dnsm_data` (same as
`rates_analysis_productive_w_thrifty_multi.ipynb`), filter to site 9
IGHV1 A→G with germline codon and non-leaf, then count per V gene.


In [4]:
# Load observed data (same pipeline as rates_analysis_productive_w_thrifty_multi)
all_site_sub_probs = []
all_pcp_dfs = []
pcp_offset = 0

for ds in DATASETS:
    print(f"Loading observed data for {ds}...")
    site_df, pcp_df_single = load_and_process_dnsm_data(
        model_name="dnsm_1m-v1jaffe+v1tang-joint",
        dataset_name=ds, numbering_scheme=NUMBERING_SCHEME
    )
    site_df['pcp_index'] = site_df['pcp_index'] + pcp_offset
    pcp_df_single = pcp_df_single.copy()
    pcp_df_single.index = pcp_df_single.index + pcp_offset
    pcp_offset += len(pcp_df_single)
    all_site_sub_probs.append(site_df)
    all_pcp_dfs.append(pcp_df_single)

site_sub_probs_df_germline_total = pd.concat(all_site_sub_probs, ignore_index=True)
total_pcp_df = pd.concat(all_pcp_dfs)


Loading observed data for v1jaffe...


Loading observed data for v1tang...


In [5]:
# Filter to site 9, IGHV1, germline codon, non-leaf, parent AA = A
pcp_indices_non_leaf = total_pcp_df[~total_pcp_df['child_is_leaf']].index

site9_obs = site_sub_probs_df_germline_total[
    (site_sub_probs_df_germline_total['site'] == TARGET_SITE) &
    (site_sub_probs_df_germline_total['v_family'] == TARGET_V_FAMILY) &
    (site_sub_probs_df_germline_total['is_germline_codon'] == True) &
    (site_sub_probs_df_germline_total['parent_aa'] == TARGET_PARENT_AA) &
    (site_sub_probs_df_germline_total['pcp_index'].isin(pcp_indices_non_leaf))
].copy()

# Count A->G mutations per V gene
site9_obs['is_target_mutation'] = (
    (site9_obs['child_aa'] == TARGET_CHILD_AA) &
    (site9_obs['mutation'] == True)
)

by_vgene_observed = site9_obs.groupby('v_gene').agg(
    n_pcps_observed=('pcp_index', 'count'),
    observed=('is_target_mutation', 'sum'),
)

print(f"Total observed A\u2192G mutations at site 9 (IGHV1): {by_vgene_observed['observed'].sum():.0f}")
print(f"\nPer-V-gene observed counts:")
print(by_vgene_observed.sort_values('observed', ascending=False).to_string())

Total observed A→G mutations at site 9 (IGHV1): 171

Per-V-gene observed counts:
               n_pcps_observed  observed
v_gene                                  
IGHV1-18*01               4419       132
IGHV1-18*04                735        23
IGHV1-2*02                4665         7
IGHV1-3*01                3052         3
IGHV1-2*04                 174         1
IGHV1-69*12                322         1
IGHV1-46*01               3097         1
IGHV1-69*01               1755         1
IGHV1-69*06                395         1
IGHV1-8*01                1524         1
IGHV1-2*06                 167         0
IGHV1-3*02                  18         0
IGHV1-24*01                606         0
IGHV1-69*02                122         0
IGHV1-46*04                287         0
IGHV1-46*03                416         0
IGHV1-69*04                191         0
IGHV1-69*09                105         0
IGHV1-69*05                 29         0
IGHV1-69*13                224         0
IGHV1-69-2*01    

## 3. Expected counts per V gene at site 9

Load Thrifty neutral expected counts (same as
`rates_analysis_productive_w_thrifty_multi.ipynb`), filter to site 9
IGHV1 A→G with germline codon and non-leaf, then sum per V gene.


In [6]:
# Load Thrifty neutral expected data
all_aa_neutral = []
all_neutral_pcp_dfs = []
pcp_offset = 0

for ds in DATASETS:
    print(f"Loading expected data for {ds}...")
    neutral_ds = CachedNeutralMutabilityDataset(
        dataset_nickname=ds,
        branch_length_mode=BRANCH_LENGTH_METHOD,
        branch_length_scale_factor=BRANCH_LENGTH_SCALE_FACTOR,
        numbering_scheme=NUMBERING_SCHEME,
        skip_nucleotide=True,
    )
    neutral_ds.pcp_df = neutral_ds.pcp_df.copy()
    neutral_ds.pcp_df.index = neutral_ds.pcp_df.index + pcp_offset
    neutral_ds.aa_neutral_df['pcp_index'] = neutral_ds.aa_neutral_df['pcp_index'] + pcp_offset
    all_aa_neutral.append(neutral_ds.aa_neutral_df)
    all_neutral_pcp_dfs.append(neutral_ds.pcp_df)
    pcp_offset += len(neutral_ds.pcp_df)

neutral_pcp_df = pd.concat(all_neutral_pcp_dfs)
aa_neutral_df = pd.concat(all_aa_neutral, ignore_index=True)
aa_neutral_df = add_germline_information(neutral_pcp_df, aa_neutral_df, numbering_scheme=NUMBERING_SCHEME)


Loading expected data for v1jaffe...
Loading NeutralMutabilityDataset data from gzip cache...


✓ Loaded from gzip cache:
  - Nucleotide DataFrame: skipped
  - Amino Acid DataFrame: 195,090,853 rows
  - Amino Acid to Any DataFrame: 27,803,432 rows
  - Codon DataFrame: 250,230,888 rows
  - Codon to Any DataFrame: 27,803,432 rows
  - PCP DataFrame: 228,789 rows


Loading expected data for v1tang...
Loading NeutralMutabilityDataset data from gzip cache...


✓ Loaded from gzip cache:
  - Nucleotide DataFrame: skipped
  - Amino Acid DataFrame: 445,297,648 rows
  - Amino Acid to Any DataFrame: 63,422,647 rows
  - Codon DataFrame: 570,803,823 rows
  - Codon to Any DataFrame: 63,422,647 rows
  - PCP DataFrame: 522,580 rows


In [7]:
# Filter to site 9, IGHV1, A->G, germline codon, non-leaf
pcp_indices_non_leaf_neutral = neutral_pcp_df[~neutral_pcp_df['child_is_leaf']].index

site9_expected = aa_neutral_df[
    (aa_neutral_df['site'] == TARGET_SITE) &
    (aa_neutral_df['v_family'] == TARGET_V_FAMILY) &
    (aa_neutral_df['current_aa'] == TARGET_PARENT_AA) &
    (aa_neutral_df['transition_aa'] == TARGET_CHILD_AA) &
    (aa_neutral_df['is_germline_codon'] == True) &
    (aa_neutral_df['pcp_index'].isin(pcp_indices_non_leaf_neutral))
]

by_vgene_expected = site9_expected.groupby('v_gene').agg(
    n_pcps_expected=('pcp_index', 'count'),
    expected=('substitution_probability', 'sum'),
)

print(f"Total expected A→G at site 9 (IGHV1): {by_vgene_expected['expected'].sum():.2f}")
print(f"\nPer-V-gene expected counts:")
print(by_vgene_expected.sort_values('expected', ascending=False).to_string())


Total expected A→G at site 9 (IGHV1): 169.74

Per-V-gene expected counts:
               n_pcps_expected   expected
v_gene                                   
IGHV1-18*01               4419  74.682625
IGHV1-2*02                4665  22.925837
IGHV1-46*01               3097  15.295612
IGHV1-3*01                3052  12.731440
IGHV1-18*04                735  12.608829
IGHV1-69*01               1755   8.640372
IGHV1-8*01                1524   6.160241
IGHV1-24*01                606   2.664780
IGHV1-69*12                322   2.387409
IGHV1-46*03                416   2.335244
IGHV1-69*06                395   2.283503
IGHV1-46*04                287   1.393340
IGHV1-69*13                224   1.378673
IGHV1-69*04                191   0.956752
IGHV1-2*06                 167   0.685280
IGHV1-69*02                122   0.625451
IGHV1-69*09                105   0.617764
IGHV1-2*04                 174   0.606597
IGHV1-69-2*01               84   0.506000
IGHV1-69*05                 29   0.166179
IG

## 4. Combined view and rate ratios with/without IGHV1-18

Merge observed, expected, and DASM selection factors per V gene.
Then compute aggregate log(observed/expected) rate ratios with and without IGHV1-18.


In [8]:
# Merge all tables
combined = by_vgene_expected[['expected']].merge(
    by_vgene_observed[['observed']],
    left_index=True, right_index=True, how='outer'
).merge(
    by_vgene_dasm[['median_log_sf']],
    left_index=True, right_index=True, how='outer'
).fillna({'observed': 0})
combined = combined.sort_values('expected', ascending=False, na_position='last')

total_expected = combined['expected'].sum()
total_observed = combined['observed'].sum()
combined['pct_expected'] = 100 * combined['expected'] / total_expected
combined['pct_observed'] = 100 * combined['observed'] / total_observed

print("Per-V-gene breakdown: observed, expected, and DASM log SF")
print(combined[['observed', 'pct_observed', 'expected', 'pct_expected', 'median_log_sf']].to_string())
print(f"\nTotals: observed={total_observed:.0f}, expected={total_expected:.2f}")


Per-V-gene breakdown: observed, expected, and DASM log SF
               observed  pct_observed   expected  pct_expected  median_log_sf
v_gene                                                                       
IGHV1-18*01       132.0     77.192982  74.682625     43.996986       0.942035
IGHV1-2*02          7.0      4.093567  22.925837     13.506056      -1.159822
IGHV1-46*01         1.0      0.584795  15.295612      9.010943      -1.134124
IGHV1-3*01          3.0      1.754386  12.731440      7.500339      -1.037322
IGHV1-18*04        23.0     13.450292  12.608829      7.428106       1.026230
IGHV1-69*01         1.0      0.584795   8.640372      5.090211      -0.972909
IGHV1-8*01          1.0      0.584795   6.160241      3.629118      -1.334214
IGHV1-24*01         0.0      0.000000   2.664780      1.569874      -1.189354
IGHV1-69*12         1.0      0.584795   2.387409      1.406469      -0.893494
IGHV1-46*03         0.0      0.000000   2.335244      1.375737      -1.052731
IGHV1-

In [9]:
# Rate ratios: all V genes vs excluding IGHV1-18
is_ighv1_18 = combined.index.str.startswith('IGHV1-18')

results = {}
for label, mask in [('All IGHV1', slice(None)), ('Excluding IGHV1-18', ~is_ighv1_18), ('IGHV1-18 only', is_ighv1_18)]:
    obs = combined.loc[mask, 'observed'].sum()
    exp = combined.loc[mask, 'expected'].sum()
    log_ratio = np.log((obs + PSEUDOCOUNT) / (exp + PSEUDOCOUNT))
    results[label] = {'observed': obs, 'expected': exp, 'log_ratio': log_ratio}
    print(f"=== {label} ===")
    print(f"  Observed: {obs:.0f}")
    print(f"  Expected: {exp:.2f}")
    print(f"  Log(obs/exp): {log_ratio:.4f}")
    print()

print(f"DASM median log SF (all V genes): {dasm_site9['log_selection_factor'].median():.4f}")
print(f"\nExcluding IGHV1-18 brings the rate ratio in line with DASM's prediction of purifying selection.")


=== All IGHV1 ===
  Observed: 171
  Expected: 169.74
  Log(obs/exp): 0.0073

=== Excluding IGHV1-18 ===
  Observed: 16
  Expected: 82.45
  Log(obs/exp): -1.6149

=== IGHV1-18 only ===
  Observed: 155
  Expected: 87.29
  Log(obs/exp): 0.5717

DASM median log SF (all V genes): -1.0276

Excluding IGHV1-18 brings the rate ratio in line with DASM's prediction of purifying selection.
